# 02 — Skeleton Extraction: ShuttleSet

Batch-process ShuttleSet match frames through YOLOv8-Pose to extract dual-player skeletons.

For each shot record, a `shot_window`-frame window centred on `frame_num` is extracted,
skeletons are Y-sorted, then optionally hitter-first reordered using `player_location_y`.

**Player ordering convention:**
- **Player 0** = top-court (smaller Y centroid)
- **Player 1** = bottom-court (larger Y centroid)
- Hitter-first reorder uses `player_location_y` from ShuttleSet JSON records.

**Steps:**
1. (Optional) Clean old ShuttleSet skeletons
2. Extract per-shot skeletons for all matches
3. Verify Y-sort ordering
4. Test ShuttleSet dataset loading

In [17]:
!pip install tqdm

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import cv2
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.config import *
from src.config import get_config
# from src.data.pose_extractor import PoseExtractor # Uncomment if using the config file 
from src.data.graph_builder import GraphBuilder # Uncomment if using the config file
from src.data.feature_eng import FeatureEngineer # Uncomment if using the config file

cfg = get_config()

## 0. (Optional) Cleanup

Run this cell only if ShuttleSet skeletons need to be re-extracted.

In [ ]:
import shutil
FORCE_REEXTRACT = False  # ← set True to delete and re-extract

if FORCE_REEXTRACT:
    ss_npy  = list(SS_SKELETONS.glob('*.npy'))
    ss_dirs = [d for d in SS_SKELETONS.iterdir() if d.is_dir()] if SS_SKELETONS.exists() else []
    for f in ss_npy:  f.unlink()
    for d in ss_dirs: shutil.rmtree(d)
    print(f"Deleted {len(ss_npy)} per-match files, {len(ss_dirs)} per-shot dirs")
else:
    print("Skipping cleanup — existing skeletons retained.")


## MVP Mode

Set `MVP_MODE = True` to extract just a few rallies/matches and verify the full
pipeline before committing to the full ~2.5 h extraction.

In [5]:
MVP_MODE    = True   # ← False to process all matches
MVP_MATCHES = 1      # matches to extract in MVP mode

if MVP_MODE:
    print(f"MVP mode ON — extracting {MVP_MATCHES} match(es)")
else:
    print("Full extraction mode — processing all matches")


MVP mode ON — extracting 3 FB rallies, 1 SS matches


## 2. Extract ShuttleSet Skeletons (per-shot)

For each shot record in the JSON outputs, extract a `shot_window`-frame window
centred on `frame_num` from the match frames, apply **hitter-first reordering**
using `player_location_y`, and save as a per-shot `.npy` file.

This mirrors FineBadminton's per-rally approach so both datasets are consistent.

In [25]:
from src.data.dataset import _reorder_hitter_first_by_location

SS_SKELETONS.mkdir(parents=True, exist_ok=True)

# Load all shot records grouped by match
match_records = {}   # match_id → list of records
for json_file in sorted(SS_OUTPUTS.glob('*.json')):
    with open(json_file) as f:
        records = json.load(f)
    if records:
        match_id = records[0].get('match_id', json_file.stem)
        match_records[match_id] = records

print(f"Matches with JSON records: {len(match_records)}")

# Apply MVP limit
match_ids = sorted(match_records.keys())
if MVP_MODE:
    match_ids = match_ids[:MVP_MATCHES]
    print(f"MVP: processing {len(match_ids)} match(es) → {match_ids}")

for match_id in match_ids:
    frame_count = len(list((SS_FRAMES / match_id).glob('*.png'))) if (SS_FRAMES / match_id).exists() else 0
    shot_count = len(match_records[match_id])
    print(f"  {match_id[:50]}: {frame_count} frames, {shot_count} shots")

Matches with JSON records: 25
MVP: processing 1 match(es) → ['An_Se_Young_Ratchanok_Intanon_YONEX_Thailand_Open_2021_QuarterFinals']
  An_Se_Young_Ratchanok_Intanon_YONEX_Thailand_Open_: 1972 frames, 663 shots


In [26]:
HALF = cfg.data.shot_window // 2   # frames before/after hit frame

for match_id in tqdm(match_ids, desc='SS Matches'):
    match_frame_dir = SS_FRAMES / match_id
    match_skel_dir  = SS_SKELETONS / match_id
    match_skel_dir.mkdir(parents=True, exist_ok=True)

    if not match_frame_dir.exists():
        print(f"  [SKIP] No frame dir: {match_id}")
        continue

    # Build frame_num → file path index for this match
    frame_index = {}
    for fp in match_frame_dir.glob('*.png'):
        try:
            fnum = int(fp.stem.replace('frame_', ''))
            frame_index[fnum] = fp
        except ValueError:
            pass

    if not frame_index:
        print(f"  [SKIP] No frames indexed: {match_id}")
        continue

    records = match_records[match_id]
    skipped, saved = 0, 0

    for record in tqdm(records, desc=f'  {match_id[:30]}', leave=False):
        rally   = record.get('rally', 0)
        ball_rnd = record.get('ball_round', 0)
        shot_file = match_skel_dir / f"r{rally:04d}_b{ball_rnd:04d}.npy"

        if shot_file.exists():
            saved += 1
            continue

        hit_frame = record.get('frame_num')
        if hit_frame is None:
            skipped += 1
            continue

        # Collect shot_window frames centred on hit_frame
        frame_nums = range(hit_frame - HALF, hit_frame - HALF + cfg.data.shot_window)
        frames = []
        for fn in frame_nums:
            fp = frame_index.get(fn)
            img = cv2.imread(str(fp)) if fp else None
            frames.append(img)

        # Fill missing frames by neighbour copy
        last_valid = None
        filled = []
        for img in frames:
            if img is not None:
                last_valid = img
            filled.append(last_valid if last_valid is not None else
                          np.zeros((480, 640, 3), dtype=np.uint8))

        # Batched extraction via extract_sequence (Y-sorted, forward-fill built in)
        # Returns (2, T, 34) — already in model format
        raw_skel = extractor.extract_sequence(filled)

        # Hitter-first reordering using annotated player Y position
        player_loc_y = record.get('player_location_y')
        if player_loc_y is not None and not (isinstance(player_loc_y, float) and np.isnan(player_loc_y)):
            raw_skel = _reorder_hitter_first_by_location(raw_skel, float(player_loc_y))

        np.save(shot_file, raw_skel)
        saved += 1

    print(f"  {match_id[:40]}: saved={saved}, skipped={skipped}")

total_shots = len(list(SS_SKELETONS.rglob('*.npy')))
print(f"\nTotal per-shot ShuttleSet skeletons: {total_shots}")

SS Matches: 100%|██████████| 1/1 [6:07:43<00:00, 22063.43s/it]

  An_Se_Young_Ratchanok_Intanon_YONEX_Thai: saved=663, skipped=0

Total per-shot ShuttleSet skeletons: 460


## 3. Verify Y-Sort and Hitter-First Reordering

Checks both FineBadminton and ShuttleSet skeletons:
- **Y-sort**: player 0 should always have a smaller mean Y than player 1
- **Hitter-first**: after `FineBadmintonDataset` loads a shot, the hitting player should be at nodes 0–16

In [20]:
# ── ShuttleSet Y-sort check ──────────────────────────────────────────────────
ss_skel_files = sorted(SS_SKELETONS.rglob('*.npy'))
print(f"ShuttleSet skeletons: {len(ss_skel_files)}")
if ss_skel_files:
    ok = fail = 0
    for f in ss_skel_files:
        skel = np.load(f)
        p0_y = skel[1, :, :17].mean()
        p1_y = skel[1, :, 17:].mean()
        if p0_y < p1_y: ok += 1
        else: fail += 1
    print(f"  Y-sort: {ok} OK, {fail} FAIL out of {ok+fail}")
else:
    print("  No ShuttleSet skeletons found — run extraction cell first.")


FineBadminton skeletons: 3
Y-sort check (player 0 should be SMALLER Y than player 1):
  [OK] 0011_001: p0_Y=263.6, p1_Y=402.5
  [OK] 0011_002: p0_Y=269.1, p1_Y=405.6
  [OK] 0011_003: p0_Y=278.2, p1_Y=431.2

ShuttleSet skeletons: 0
  (no ShuttleSet skeletons yet — run Section 2 first)

Hitter-first check via FineBadmintonDataset (L0 = raw x,y for easy reading):
[INFO] FineBadminton: 296 labeled shots across 40 rallies
  intercept: 108
  create_depth: 61
  defensive: 59
  passive: 50
  move_to_net: 18
Samples with real skeletons: 42
  [OK] sample 0: hitter='bottom', p0_Y=413.8, p1_Y=293.5, label=create_depth
  [OK] sample 1: hitter='top', p0_Y=251.4, p1_Y=438.3, label=intercept
  [OK] sample 2: hitter='bottom', p0_Y=344.3, p1_Y=241.8, label=defensive
  [OK] sample 3: hitter='top', p0_Y=219.7, p1_Y=310.1, label=move_to_net
  [OK] sample 4: hitter='bottom', p0_Y=309.7, p1_Y=228.9, label=intercept


## 6. Test Dataset Loading

In [14]:
from src.data.dataset import ShuttleSetDataset

ss_ds = ShuttleSetDataset(feature_layer='L2')
print(f"ShuttleSet samples: {len(ss_ds)}")

if len(ss_ds) > 0:
    sample = ss_ds[0]
    if isinstance(sample, tuple):
        x, st = sample
        print(f"  Sample shape: {x.shape}, shot_type: {st}")
    else:
        print(f"  Sample shape: {sample.shape}")


[INFO] FineBadminton: 296 labeled shots across 40 rallies
  intercept: 108
  create_depth: 61
  defensive: 59
  passive: 50
  move_to_net: 18

FineBadminton samples: 296
  Sample shape: torch.Size([9, 16, 34]), label: 3 (create_depth)
  Has real data: True
[INFO] ShuttleSet: 21191 shot records from 25 matches (skeletons pending extraction)

ShuttleSet samples: 21191
  Sample shape: torch.Size([9, 16, 34]), shot_type: 1
